In [ ]:
import ansys.aedt.core
import os
import tempfile
import time
from ansys.aedt.core import Maxwell3d

AEDT_VERSION = "2024.2"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.

In [ ]:
"D:\EM_KDH\Moa_Edu_Sim\Basic"

In [3]:
baseDir="D:\KangDH\deVSimulation\Ansys"
project_name = os.path.join(baseDir, "PlanarCapacitor.aedt")

In [67]:
0.0058

0.0058

# 2D

In [ ]:
from ansys.aedt.core import Maxwell2d

m2d = Maxwell2d(
    project=project_name,
    version=AEDT_VERSION,
    new_desktop=False,
    non_graphical=NG_MODE,
)
rect1=m2d.modeler.create_rectangle([0, 0], [10, 10], name="rect1", matname="copper")
rect1.id
m2dModel=m2d.modeler
m2dModel.object_list   # List of objects in the model
m2dModel.object_names
rect1_1=m2dModel.get_object_from_name("rect1_1")
rect1=m2dModel.get_object_from_name("rect1")
rect1.color='Red'
Vertrect=rect1.vertices
rect1.fillet(Vertrect[1],None,0.5,0)


# 3D 

In [ ]:
m3d = Maxwell3d(
    project=project_name,
    version=AEDT_VERSION,
    new_desktop=False,
    non_graphical=NG_MODE,
)
m3dOdesign=m2d.odesign

In [ ]:
m3dModel=m3d.modeler

 ## Electrostatics

### 3.1 Planar Capacitor

In [26]:
curObList = m3dModel.object_list

downPlate = next((obj for obj in curObList if obj.name == "downPlate"), None)
if downPlate is None:
    downPlate = m3dModel.create_box([0, 0, 0], [25, 25, 2], name="downPlate", material="copper")
upPlate = next((obj for obj in curObList if obj.name == "upPlate"), None)
if upPlate is None:
    upPlate = m3dModel.create_box([0, 0, 3], [25, 25, 2], name="upPlate", material="copper")

In [ ]:
downPlate.color='Red'
upPlate.color='Red'
Gap=m3dModel.create_box([0,0,2],[25,25,1],name="Gap",material="vacuum")
Gap.color='Gray'

#### assign Voltage & Matrix

In [ ]:
downVoltage = next((voltage for voltage in m3d.boundaries if voltage.name == "downVoltage"), None)
if downVoltage is None:
    downVoltage = m3d.assign_voltage(downPlate, 0, "downVoltage")

upVoltage = next((voltage for voltage in m3d.boundaries if voltage.name == "upVoltage"), None)
if upVoltage is None:
    upVoltage = m3d.assign_voltage(upPlate, 0, "upVoltage")
m3d.assign_matrix(["downVoltage","upVoltage"],'Matrix1')

#### assign sol Setup

In [32]:
setup_name = "Setup1"
existing_setups = m3d.existing_analysis_setups
if setup_name not in existing_setups:
    solSetupObj = m3d.create_setup(setup_name)
else:
    solSetupObj = m3d.get_setup(setup_name)
solSetupObj.available_properties
solSetupObj.props["MaximumPasses"]=10
solSetupObj.props["PercentError"]=1
solSetupObj.props["Enabled"]=1
solSetupObj.props["SolveMatrixAtLast"]="True"
solSetupObj.props["PercentRefinement"]=50
solSetupObj.props["MinimumPasses"]=2
solSetupObj.props["MinimumConvergedPasses"]=1


In [ ]:
m3d.valid_design
m3dAnalyObj=m3d.analyze()

#### get Solution

In [ ]:
availabeQ=m3d.post.available_report_quantities()
oModule=m3dOdesign.GetModule("AnalysisSetup")
oModule.ExportSolnData("Setup1 : LastAdaptive", "Matrix1", False, "", "C:/Users/user/Downloads/PlanarCapacitor_Maxwell 3D_8OW.txt")

with open("C:/Users/user/Downloads/PlanarCapacitor_Maxwell 3D_8OW.txt", "r") as file:
 lines = file.readlines()

In [ ]:
lines

### 3.2 Cylindrical Capacitor


In [152]:
baseDir="D:\KangDH\deVSimulation\Ansys"
project_name = os.path.join(baseDir, "CylinderCapacitor.aedt")

In [ ]:
m3dCylinder = Maxwell3d(
    project=project_name,
    version=AEDT_VERSION,
    new_desktop=False,
    non_graphical=NG_MODE,
)


In [155]:
m3dCylyderOdesign=m3dCylinder.odesign
m3dCylyderOdesign.SetSolutionType("Electrostatic")

#### geometry

In [ ]:
cylinderInnerObj = next((obj for obj in m3dCylinder.modeler.primitives.object_list if obj.name == "Inner"), None)
if cylinderInnerObj is None:
    cylinderInnerObj = m3dCylinder.modeler.primitives.create_cylinder([0, 0, 0],
                         [0, 0, -4],
                         radius=0.6,
                         height=25,
                         num_sides=24,
                         name="Inner",
                         material="copper")

cylinderOuterObj = next((obj for obj in m3dCylinder.modeler.primitives.object_list if obj.name == "Outer"), None)
if cylinderOuterObj is None:
    cylinderOuterObj = m3dCylinder.modeler.primitives.create_cylinder([0, 0, 0],
                         [0, 0, -4],
                         radius=1.2,
                         height=25,
                         num_sides=24,
                         name="Outer",
                         material="copper")

VaccumObj = next((obj for obj in m3dCylinder.modeler.primitives.object_list if obj.name == "Gap"), None)
if VaccumObj is None:
    VaccumObj = m3dCylinder.modeler.primitives.create_cylinder([0, 0, 0],
                         [0, 0, -4],
                         radius=1,
                         height=25,
                         num_sides=24,
                         name="Gap",
                         material="vacuum")


#### m3d Modeler

In [ ]:
m3dCylinderModel=m3dCylinder.modeler
m3dCylinderModel.subtract([cylinderOuterObj.id], [VaccumObj.id], False)

RegionObj = next((obj for obj in m3dCylinder.modeler.primitives.object_list if obj.name == "Region"), None)
if RegionObj is None:
    RegionObj = m3dCylinderModel.create_region(
        pad_value=[300, 300, 300, 300, 0, 0],
        pad_type='Percentage Offset',
        name='Region',
        material='Vacuum',
    )


#### Assign

In [ ]:
m3dCylinder.assign_material(["Inner"],'copper')
m3dCylinder.assign_material(["Outer"],'copper')


#### assign voltage

In [ ]:
m3dCylinder.assign_voltage('I')

In [164]:
innerVoltage = next((voltage for voltage in m3dCylinder.boundaries if voltage.name == "InnerVoltage"), None)
if innerVoltage is None:
    innerVoltage = m3dCylinder.assign_voltage("Inner", 0, "InnerVoltage")

outerVoltage = next((voltage for voltage in m3dCylinder.boundaries if voltage.name == "OuterVoltage"), None)
if outerVoltage is None:
    outerVoltage = m3dCylinder.assign_voltage("Outer", 1000, "OuterVoltage")

m3dCylinder.assign_matrix(["InnerVoltage","OuterVoltage"],'Matrix1')
m3dCylinder.assign_force(["Inner"],coordinate_system='Global',is_virtual=True,force_name='Force1')
force1 = next((force for force in m3dCylinder.boundaries if force.name == "Force1"), None)
if force1 is None:
    force1 = m3dCylinder.assign_force(["Inner"], coordinate_system='Global', is_virtual=True, force_name='Force1')

#### Setup

In [165]:
setup_name = "Setup1"
existing_setups = m3dCylinder.existing_analysis_setups
if setup_name not in existing_setups:
    solCylinder3dSetupObj = m3dCylinder.create_setup(setup_name)
else:
    solCylinder3dSetupObj = m3dCylinder.get_setup(setup_name)
solCylinder3dSetupObj.props["MaximumPasses"]=10
solCylinder3dSetupObj.props["PercentError"]=1
solCylinder3dSetupObj.props["Enabled"]=1
solCylinder3dSetupObj.props["SolveMatrixAtLast"]="True"
solCylinder3dSetupObj.props["PercentRefinement"]=50
solCylinder3dSetupObj.props["MinimumPasses"]=2
solCylinder3dSetupObj.props["MinimumConvergedPasses"]=1

In [ ]:
m3dCylinder.valid_design
m3dAnalyObj=m3dCylinder.analyze()

#### Post

In [166]:
import ansys.aedt.core.visualization.post.common

##### def create_3d_plot

In [ ]:
m3CylinderPost=m3dCylinder.post
m3CylinderPost.plot_field('Mag_E','Region','Volume')

##### def get_solution_data

In [ ]:
solCylinderPost.export_data_to_csv('D:\EM_KDH\Moa_Edu_Sim\Basic/test.csv')
m3CylinderPost.export_field_plot(plot_name='Mag_E2',output_dir='D:\EM_KDH\Moa_Edu_Sim\Basic')
a=m3CylinderPost.ofieldsreporter
m3CylinderPost.plot_field_from_fieldplot(plot_name='E',project_path="",mesh_plot=True,image_format='png',view='isometric',plot_label='',show=True,)

### 3. 4 Spiral Coil

In [ ]:
Spiral_name = os.path.join(baseDir, "SpiralCoil.aedt")
m3dSpiralCoil = Maxwell3d(
    project=Spiral_name,
    version=AEDT_VERSION,
    new_desktop=False,
    non_graphical=NG_MODE,
)
m3dOdesign=m3dSpiralCoil.odesign


In [313]:
m3dOdesign.SetSolutionType("EddyCurrent")

In [314]:
oEditor=m3dSpiralCoil.oeditor

In [315]:
oEditor.SetModelUnits(
	[
		"NAME:Units Parameter",
		"Units:="		, "cm",
		"Rescale:="		, False,
		"Max Model Extent:="	, 10000
	])

In [ ]:
m3dModel=m3dSpiralCoil.modeler
curObList = m3dModel.object_list
display(curObList)

#### Disk

In [ ]:
DiskObj = next((obj for obj in curObList if obj.name == "Disk"), None)
if DiskObj is None:
    DiskObj=m3dModel.create_polyhedron(origin=[41,0,1.5],center=[0,0,1.5],name='Disk',height=1,material='cast_iron',num_sides=36)
    # upPlate = m3dModel.create_box([0, 0, 3], [25, 25, 2], name="upPlate", material="copper")

#### Coil

In [319]:
HelixDObj = next((obj for obj in curObList if obj.name == "Coil"), None)
if HelixDObj is None:
     HelixDObj=m3dModel.create_udp(dll="SegmentedHelix/PolygonHelix.dll",parameters=[["PolygonSegment","4"], ["PolygonRadius","1.5cm"],["StartHelixRadius","15cm"],["RadiusChange","3.1cm"],["Pitch","0mm"],["Turns","8"],["SegmentsPerTurn","36"],["RightHanded","1"]],library="syslib",name='Coil')


oEditor.AssignMaterial(
	[
		"NAME:Selections",
		"AllowRegionDependentPartSelectionForPMLCreation:=", True,
		"AllowRegionSelectionForPMLCreation:=", True,
		"Selections:="		, "Coil"
	], 
	[
		"NAME:Attributes",
		"MaterialValue:="	, "\"copper\"",
		"SolveInside:="		, True,
		"ShellElement:="	, False,
		"ShellElementThickness:=", "nan ",
		"ReferenceTemperature:=", "nan ",
		"IsMaterialEditable:="	, True,
		"IsSurfaceMaterialEditable:=", True,
		"UseMaterialAppearance:=", False,
		"IsLightweight:="	, False
	])


#### connection 

In [ ]:
Box1Obj = next((obj for obj in curObList if obj.name == "Box1"), None)
if Box1Obj is None:
    Box1Obj=m3dModel.create_box([14,0,-2],[2,2,-2],name='Box1',material='copper')
Box2Obj = next((obj for obj in curObList if obj.name == "Box2"), None)
if Box2Obj is None:
    Box2Obj=m3dModel.create_box([40.5,0,-2],[-2,-2,-2],name='Box2',material='copper')


In [ ]:
m3dModel.create_object_from_face(assignment=3732)

#### Dupliate

In [ ]:
m3dModel.duplicate_along_line(assignment=["Box1","Box2"],vector=[0,0,1])


#### Section for Coil


In [9]:
m3dModel.section(assignment="Coil",plane="YZ")
CoilTerminal=m3dModel.sheet_objects[0]
CoilTerminal.name='Coil_Terminal'

#### Unite

In [ ]:
m3dModel.unite(assignment=["Box1","Box1_1","Box2","Box2_1","Coil","Box1_ObjectFromFace1"])

#### Boolen

In [ ]:
m3dModel.separate_bodies(assignment='Coil_Terminal')
sheetObjList=m3dModel.sheet_objects
m3dModel.delete(sheetObjList[2:])

#### assign Current Excitation

In [ ]:
m3dSpiralCoil.assign_current(name='I_Coil',amplitude='125A',phase='0deg',solid=True,assignment='Coil_Terminal')

In [27]:
DiskfaceList=m3dModel.create_face_list(assignment='Disk')

#### create sheet Layers on Disk

In [34]:
DiskfaceList.props
m3dModel.create_object_from_face(assignment=8)
m3dModel.move(assignment='Disk_ObjectFromFace1',vector=[0,0,0.125])

In [36]:
m3dModel.duplicate_along_line(assignment='Disk_ObjectFromFace1',vector=[0,0,0.125])

(True, ['Disk_ObjectFromFace1_1'])

#### Create Region

In [39]:
m3dModel.create_cylinder(origin=[0,0,-50],orientation=[0,0,0],radius=150,height=100,num_sides=36,material='vacuum',name='Region')


PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec


#### Setup


In [40]:
m3dSpiralCoil.eddy_effects_on(assignment=['Disk'])

True

In [58]:
setup_name = "Setup1"
existing_setups = m3dSpiralCoil.existing_analysis_setups
if setup_name not in existing_setups:
    solSetupObj = m3dSpiralCoil.create_setup(setup_name)
else:
    solSetupObj = m3dSpiralCoil.get_setup(setup_name)
solSetupObj.available_properties
solSetupObj.props["MaximumPasses"]=15
solSetupObj.props["PercentError"]=2
solSetupObj.props["Enabled"]=1
solSetupObj.props["SolveMatrixAtLast"]="True"
solSetupObj.props["PercentRefinement"]=20
# solSetupObj.props["MinimumPasses"]=2
solSetupObj.props["MinimumConvergedPasses"]=1 

In [60]:
solSetupObj.update_property(prop_name='Frequency Setup',prop_value='500Hz')

In [61]:
m3dSpiralCoil.valid_design
m3dSpiralAnalyObj=m3dSpiralCoil.analyze()

PyAEDT INFO: Key Desktop/ActiveDSOConfigurations/Maxwell 3D correctly changed.
PyAEDT INFO: Solving all design setups.
PyAEDT INFO: Design setup None solved correctly in 0.0h 3.0m 42.0s


In [66]:
m3dSpiralCoil.mesh

## Day 2

### 3.3 Parallel Plate Capacitor (Battery)

In [ ]:
baseDir
project_name = os.path.join(baseDir, "Battery.aedt")

In [ ]:
m3dBattery=Maxwell3d(
    project=project_name,
    version=AEDT_VERSION,
    new_desktop=False,
    non_graphical=NG_MODE,
)

#### Create UDP object

In [254]:
m3dBatteryModel=m3dBattery.modeler
# helixBattery=m3dBatteryModel.create_udp(dll="RectHelix.dll",parameters=[""],library="syslib")
HelixD=m3dBatteryModel.create_udp(dll="SegmentedHelix/RectHelix.dll",parameters=[["RectHeight","75mm"], ["RectWidth","2mm"],["StartHelixRadius","12mm"],["RadiusChange","6mm"],["Pitch","0mm"],["Turns","2.5"],["SegmentsPerTurn","0"],["RightHanded","1"]],library="syslib",name='RectHelixD')

#### change Material & Name of UDP

In [ ]:
oEditor=m3dBatteryModel.oeditor

In [ ]:
oEditor.ChangeProperty(
	[
		"NAME:AllTabs",
		[
			"NAME:Geometry3DAttributeTab",
			[
				"NAME:PropServers", 
				"RectHelixD_1"
			],
			[
				"NAME:ChangedProps",
				[
					"NAME:Name",
					"Value:="		, "Electrode_A"
     
				]
			]
		]
	])
oEditor.AssignMaterial(
	[
		"NAME:Selections",
		"AllowRegionDependentPartSelectionForPMLCreation:=", True,
		"AllowRegionSelectionForPMLCreation:=", True,
		"Selections:="		, "Electrode_A"
	], 
	[
		"NAME:Attributes",
		"MaterialValue:="	, "\"aluminum\"",
		"SolveInside:="		, True,
		"ShellElement:="	, False,
		"ShellElementThickness:=", "nan ",
		"ReferenceTemperature:=", "nan ",
		"IsMaterialEditable:="	, True,
		"IsSurfaceMaterialEditable:=", True,
		"UseMaterialAppearance:=", False,
		"IsLightweight:="	, False
	])


#### eleC B

In [291]:
ElecB=m3dBatteryModel.create_udp(dll="SegmentedHelix/RectHelix.dll",parameters=[["RectHeight","75mm"], ["RectWidth","2mm"],["StartHelixRadius","15mm"],["RadiusChange","6mm"],["Pitch","0mm"],["Turns","2.5"],["SegmentsPerTurn","0"],["RightHanded","1"]],library="syslib",name='Electrode_B')

In [293]:
def assign_material_and_change_color(oEditor, electrode_name, material="aluminum", color=(0, 0, 255)):
	# Assign material
	oEditor.AssignMaterial(
		[
			"NAME:Selections",
			"AllowRegionDependentPartSelectionForPMLCreation:=", True,
			"AllowRegionSelectionForPMLCreation:=", True,
			"Selections:=", electrode_name
		],
		[
			"NAME:Attributes",
			"MaterialValue:=", f"\"{material}\"",
			"SolveInside:=", True,
			"ShellElement:=", False,
			"ShellElementThickness:=", "nan ",
			"ReferenceTemperature:=", "nan ",
			"IsMaterialEditable:=", True,
			"IsSurfaceMaterialEditable:=", True,
			"UseMaterialAppearance:=", False,
			"IsLightweight:=", False
		]
	)
	
	# Change color
	oEditor.ChangeProperty(
		[
			"NAME:AllTabs",
			[
				"NAME:Geometry3DAttributeTab",
				[
					"NAME:PropServers",
					electrode_name
				],
				[
					"NAME:ChangedProps",
					[
						"NAME:Color",
						"R:=", color[0],
						"G:=", color[1],
						"B:=", color[2]
					]
				]
			]
		]
	)

# Example usage
assign_material_and_change_color(oEditor, "Electrode_B", color=(0, 0, 255))
assign_material_and_change_color(oEditor, "Electrode_A", color=(255, 0, 0))


<!-- #### Duplicate -->